## Terminology Update

Some historical output logs in this notebook were captured before the current naming migration.
Current runtime terminology in the codebase is:
- `single_csv`
- `multi_csv`
- `ExecutionContext`


In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

print("Imports OK")

Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
metadata_standard=METADATA_STANDARDS['spatial_ecological']
orchestrator = Orchestrator(topology_name='fast')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

2026-04-23 11:33:05,750 - INFO - root - PlanExecutor initialized with topology: fast
2026-04-23 11:33:05,751 - INFO - root -   Players per step: 2
2026-04-23 11:33:05,751 - INFO - root -   Debate rounds: 1
2026-04-23 11:33:05,751 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-04-23 11:33:05,751 - INFO - root - Orchestrator initialized with topology: fast
2026-04-23 11:33:05,752 - INFO - root - ============================================================
2026-04-23 11:33:05,752 - INFO - root - GENERATING PLAN
2026-04-23 11:33:05,752 - INFO - root - Context: biota_multi
2026-04-23 11:33:05,752 - INFO - root - Context type: multi_csv
2026-04-23 11:33:05,753 - INFO - root - Classified planning type: multi_csv
2026-04-23 11:33:05,753 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-04-23 11:33:05,753 - INFO - root - Multi-CSV: True
2026-04-23 11:33:05,754 - INFO - root - ============================================================
2026-04-23 11:33:05,75

In [4]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [5]:
plan.pretty_print()

Plan Steps:
Step 0: get_field_statistics
  Rationale: Profile all three resources to extract field statistics, distributions, and data quality metrics needed for metadata generation
  Required Artifacts: {}
  Produced Artifacts: ['biota:field_stats', 'samples:field_stats', 'species:field_stats']
Step 1: get_relationships
  Rationale: Discover and validate relationships between resources to understand data structure for metadata
  Required Artifacts: {}
  Produced Artifacts: ['relationships']
Step 2: generate_metadata
  Rationale: Generate final metadata using all gathered information and the metadata standard
  Required Artifacts: {'metadata_standard': 'metadata_standard', 'biota_field_stats': 'biota:field_stats', 'samples_field_stats': 'samples:field_stats', 'species_field_stats': 'species:field_stats', 'relationships': 'relationships'}
  Produced Artifacts: ['metadata_output']


In [6]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
metadata_standard = METADATA_STANDARDS["spatial_ecological"]
executor = PlanExecutor(topology_name="fast")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name="spatial_ecological"
)

2026-04-23 11:33:15,236 - INFO - root - PlanExecutor initialized with topology: fast
2026-04-23 11:33:15,237 - INFO - root -   Players per step: 2
2026-04-23 11:33:15,237 - INFO - root -   Debate rounds: 1
2026-04-23 11:33:15,237 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-04-23 11:33:15,238 - INFO - root - Using structured output schema: SpatialEcologicalMetadata
2026-04-23 11:33:15,238 - INFO - root - ============================================================
2026-04-23 11:33:15,238 - INFO - root - STARTING PLAN EXECUTION
2026-04-23 11:33:15,238 - INFO - root - Context: biota_multi
2026-04-23 11:33:15,238 - INFO - root - Type: multi_csv
2026-04-23 11:33:15,239 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-04-23 11:33:15,239 - INFO - root - Steps: 3
2026-04-23 11:33:15,239 - INFO - root - ============================================================
2026-04-23 11:33:15,239 - INFO - root - 
2026-04-23 11:33:15,239 - INFO - root - ===========

In [7]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

{'description': 'Multi-resource dataset containing benthic macrofauna '
                'observations from Wadden Sea tidal basins, including species '
                'abundance and biomass measurements linked to sample metadata '
                'and taxonomic information. Data collected from 2008-2024 '
                'across 12 tidal basins with 51,851 samples and 391,018 biota '
                'records.',
 'format': 'CSV (multi-resource: biota.csv, samples.csv, species.csv)',
 'methods': 'Grid-based (85%) and random (15%) sampling from boat platform '
            '(93%). Tidal basin and tidal flat sampling across 12 basins and '
            '41 flats. Measurements include abundance per square meter '
            '(abundance_m2), ash-free dry mass per square meter (afdm_m2), '
            'median grain size, and percentage mud. Taxonomic identification '
            'to species level (63%) with 177 unique species across 10 '
            'taxonomic groups.',
 'spatial_coverage': {